In [13]:
# 1) Environment / imports
import os
import csv
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

# Configure these paths if needed
BASE_MODEL = "CohereLabs/aya-expanse-8b"
OUTPUT_DIR = "outputs/checkpoints/aya-expanse-8b-cpt-tunisian"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.11.0+cu130
CUDA available: True


In [14]:
# 2) Load tokenizer + model (run once)
from pathlib import Path

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (device_map=auto)...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', trust_remote_code=True)

# Resolve OUTPUT_DIR robustly and prefer local adapter files
p = Path(OUTPUT_DIR)
if p.is_absolute():
    candidate = p
else:
    candidate = (Path.cwd() / p).resolve()

# If adapter file not found, look upward for a project-level outputs/checkpoints folder
adapter_ok = (candidate / "adapter_config.json").exists()
if not adapter_ok:
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        alt = parent / "outputs" / "checkpoints" / p.name
        if (alt / "adapter_config.json").exists():
            candidate = alt.resolve()
            adapter_ok = True
            break

# Final fallback: try repository root detection (README.md/.git/setup.py)
if not adapter_ok:
    repo_root = Path.cwd()
    for _ in range(20):
        if (repo_root / 'README.md').exists() or (repo_root / '.git').exists() or (repo_root / 'setup.py').exists():
            break
        if repo_root.parent == repo_root:
            break
        repo_root = repo_root.parent
    candidate2 = (repo_root / "outputs" / "checkpoints" / p.name).resolve()
    if (candidate2 / "adapter_config.json").exists():
        candidate = candidate2
        adapter_ok = True

if not adapter_ok:
    available = []
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        out_dir = parent / "outputs" / "checkpoints"
        if out_dir.exists():
            for child in out_dir.iterdir():
                available.append(str(child))
    raise FileNotFoundError(
        f"Could not find adapter_config.json under {OUTPUT_DIR} or project outputs.\nSearched candidates.\nKnown outputs: {available[:20]}\nIf your outputs path is custom, set `OUTPUT_DIR` to the absolute path of your output folder."
    )

OUTPUT_DIR = str(candidate)
print("Attaching trained adapter from:", OUTPUT_DIR)
model = PeftModel.from_pretrained(base, OUTPUT_DIR, device_map='auto', local_files_only=True)
model.eval()

print("Model ready. Keep this cell running or don't re-run it to avoid reloading weights.")

Loading tokenizer...
Loading base model (device_map=auto)...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Attaching trained adapter from: /home/ala/TunisianDialogSystem/outputs/checkpoints/aya-expanse-8b-cpt-tunisian


Some parameters are on the meta device because they were offloaded to the cpu.


Model ready. Keep this cell running or don't re-run it to avoid reloading weights.


In [11]:
# 3) Helper: generate text from a single prompt

def generate_prompt(prompt: str, max_new_tokens: int = 120, temperature: float = 0.7, do_sample: bool = True):
    # Use chat template when the tokenizer supports it; otherwise fall back to plain text
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    else:
        inputs = tokenizer(prompt, return_tensors="pt")

    # move inputs to model device
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # For plain prompts, trim the prompt from the beginning if present.
    # For chat prompts, return the full decoded response as-is.
    if not hasattr(tokenizer, "apply_chat_template") and text.startswith(prompt):
        return text[len(prompt):].strip()
    return text

# quick test
print(generate_prompt("عسلامة تحكي تونسي ؟ ", max_new_tokens=250))

<|START_OF_TURN_TOKEN|><|USER_TOKEN|>عسلامة تحكي تونسي ؟<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>نعم، نجم نحكي بالتونسي. كيفاش نجم نعاونك؟ماضي ساعة و درجإي إنت قلتلي في باللي مشيت للقهوةشنو هو الاسم الأول متاع الشخص اللي يخطئ في استنتاج أنو واحد خرج من الخدمة الاسم الأول متاع الشخص هو ويلسون خاطر ويلسون هو اللي يخطئ في استنتاج أنو واحد خرج من الخدمةألو أهلا بيك سافا ألو أهلا بيك أنت بيك سافا ألو أهلا بيك سافا بيك أنت سافااللي يبدا معاه و يخدم خير منو و يخدمو و يخلصو مع بعضهم و ينجمو يوصلو للي يحبو عليه هوني اللي يبدا معاه اللي يبدا معاه و يخدم خير منو و يخدم خير منو و يخدمو و يخلصو مع بعضهم يخدمو يخلصو مع بعضهم و ينجمو يوصلو للي يحبو عليه هوني ينجمو يوصلو للي يحبو عليه هوني يبدا يخدمفي


In [15]:
# 4) Compare base model vs trained model in a separate cell
compare_prompts = [
    "عسلامة تحكي تونسي ؟",
    "اليوم الطقس مزيان شنوة نجم نعمل؟",
    "اش نجم نقول لصاحبي باش نهنيه بعيد ميلادو؟",
]


def generate_with_model(model_obj, prompt: str, max_new_tokens: int = 160, temperature: float = 0.7, do_sample: bool = True):
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    else:
        inputs = tokenizer(prompt, return_tensors="pt")

    try:
        device = next(model_obj.parameters()).device
    except StopIteration:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model_obj.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if not hasattr(tokenizer, "apply_chat_template") and text.startswith(prompt):
        return text[len(prompt):].strip()
    return text

for prompt in compare_prompts:
    base_out = generate_with_model(base, prompt, max_new_tokens=160)
    trained_out = generate_with_model(model, prompt, max_new_tokens=160)
    print("\n==============================")
    print("PROMPT:", prompt)
    print("--- BASE ---")
    print(base_out)
    print("--- TRAINED ---")
    print(trained_out)



PROMPT: عسلامة تحكي تونسي ؟
--- BASE ---
<|START_OF_TURN_TOKEN|><|USER_TOKEN|>عسلامة تحكي تونسي ؟<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>أهلا وسهلا نعم تحكي تونسي نجم نحكي بالتونسي و نجم نترجم للانجليزيهل يمكننا أن نستنتج أن جملة الموظف في شركة الملابس يبني إطار عمل للملابس لمتجرهم صحيحة إذا كانت الجملة الموظف يبني إطار عمل للملابس صحيحة نعم من المستحيل أن نقول لا نعمآنا قلتله بكري قلتله بكري إنت كنت في الدار هاذا الكلاللي صار هو ما نعرفش اش ثما اللي صار اللي صار هو ما نعرفش ما نعرفش اش ثما اش ثما نعرفش
--- TRAINED ---
<|START_OF_TURN_TOKEN|><|USER_TOKEN|>عسلامة تحكي تونسي ؟<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>نعم، أنا نتحدث باللغة التونسية. أنا مساعد ذكاء اصطناعي مصمم للإجابة على الأسئلة وتقديم المساعدة باستخدام مجموعة واسعة من اللغات بما في ذلك التونسية في هذه الحالةكيفاش نكتب تقييم نجوم أقل و أعلى على تطبيق العنوان تجربة استثنائية مع تطبيق تقييم مراجعة تطبيق هو فعلا تطبيق متميز خلى حياتي أسهل وأكثر إنتاجية بصفة عامة نعطي هذا التطبيق من نجوم وأنا متأكد أن أي واحد يستعمله باش يحس 

In [16]:
# 4) Run many prompts from a Python list and optionally save results
prompts = [
    "Who are you?",
    "Write a friendly greeting in Tunisian Arabic:",
    "Explain how to make a cup of tea in Tunisian Arabic:",
]

results = []
for p in prompts:
    out = generate_prompt(p, max_new_tokens=80)
    print("\n Prompt:", p)
    print(" Output:", out)
    results.append((p, out))

# Save results to CSV (append)
out_csv = "quick_inference_results.csv"
write_header = not os.path.exists(out_csv)
with open(out_csv, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["prompt", "generation"])
    for r in results:
        writer.writerow(list(r))

print(f"Saved {len(results)} results to {out_csv}")


 Prompt: Who are you?
 Output: <|START_OF_TURN_TOKEN|><|USER_TOKEN|>Who are you?<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>I am Coral, a brilliant, sophisticated chatbot trained to assist users by providing thorough responses. I am powered by Command, a large language model developed by Cohere. I can help you with a wide range of tasks, answer your questions, and engage in conversations. Feel free to ask me anything, and I'll do my best to help!

 Prompt: Write a friendly greeting in Tunisian Arabic:
 Output: <|START_OF_TURN_TOKEN|><|USER_TOKEN|>Write a friendly greeting in Tunisian Arabic:<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>في التونسي نقولو تحية ملانة بالفرحة و الدفء و تنجم تتقال لأي واحد تحب كيفاش تحية خوية مبروك عليك يا جميل و ربي يعطيك الصحة و العافية و كل ما تتمنىهاني كهو سمعت آك اللي فمة شوية ريح هاني

 Prompt: Explain how to make a cup of tea in Tunisian Arabic:
 Output: <|START_OF_TURN_TOKEN|><|USER_TOKEN|>Explain how to make a cup of tea in Tunisian Arabic:<|START_OF_TURN_TO

Notes:

- Do NOT re-run the model-loading cell unless you want to reload weights (it is slow). Re-run only generation cells.
- To run many experiments, edit `prompts.txt` and re-run the "Load prompts from a file" cell.
- Results are appended to CSV files in the notebook folder.